In [1]:
# ============================================================
# INITIALIZE RETRIEVER FOR DIAGNOSTIC TEST
# ============================================================

from pathlib import Path
import sys

# Works on every machine: find the project root (the folder that contains rag/pipeline.py)
PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "rag" / "pipeline.py").exists()
)
RAG_DIR = PROJECT_ROOT / "rag"
for _p in (PROJECT_ROOT, RAG_DIR, RAG_DIR / "retrieval", RAG_DIR / "vector_store"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker
from vector_store.Metadata_filter import MetadataFilter


# ============================================================
# PATHS
# ============================================================


# rag/embeddings/processed/chunks/chunks.json
CHUNKS_FILE = (
    RAG_DIR
    / "embeddings"
    / "processed"
    / "chunks"
    / "chunks.json"
)


# ============================================================
# CHECK CHUNKS FILE
# ============================================================

print("=" * 70)
print("DIAGNOSTIC INITIALIZATION")
print("=" * 70)

print("\nChunks file:")
print(CHUNKS_FILE)

print("\nChunks file exists:")
print(CHUNKS_FILE.exists())


# ============================================================
# INITIALIZE COMPONENTS
# ============================================================

hybrid_retriever = HybridRetriever()

reranker = CrossEncoderReranker()

metadata_filter = MetadataFilter(
    chunks_file=str(CHUNKS_FILE)
)

print("\nAll components initialized successfully.")

C:\Users\Ahmed\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DIAGNOSTIC INITIALIZATION

Chunks file:
C:\Users\Ahmed\.zcode\workspace\default\Nasa Rag SYS\rag\embeddings\processed\chunks\chunks.json

Chunks file exists:
True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  21%|██        | 42/199 [00:00<00:00, 406.91it/s]

Loading weights:  42%|████▏     | 83/199 [00:00<00:00, 369.52it/s]

Loading weights:  90%|████████▉ | 179/199 [00:00<00:00, 621.78it/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:  65%|██████▍   | 68/105 [00:00<00:00, 495.26it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 629.28it/s]

Model loaded successfully.
Loaded 722 chunks.

All components initialized successfully.


In [2]:
# Test what HybridRetriever actually returns

test_result = hybrid_retriever.search(
    query="How does oxygen concentration affect flame behavior in microgravity?",
    dense_top_k=20,
    bm25_top_k=20,
    top_k=5
)

print("=" * 70)
print("HYBRID RESULT STRUCTURE")
print("=" * 70)

print(test_result[0].keys())

print("\nFirst result:")
print(test_result[0])

HYBRID RESULT STRUCTURE
dict_keys(['chunk_id', 'rrf_score', 'dense_rank', 'bm25_rank', 'dense_score', 'bm25_score', 'hybrid_rank'])

First result:
{'chunk_id': 720, 'rrf_score': 0.03200204813108039, 'dense_rank': 2, 'bm25_rank': 3, 'dense_score': 0.7630226016044617, 'bm25_score': 6.94641823500619, 'hybrid_rank': 1}


In [3]:
# ============================================================
# TEST ENRICHMENT
# ============================================================

enriched_results = metadata_filter.enrich_results(
    test_result
)

print("=" * 70)
print("ENRICHED RESULT STRUCTURE")
print("=" * 70)

print("\nKeys:")
print(enriched_results[0].keys())

print("\nMetadata:")
print(
    enriched_results[0].get(
        "metadata",
        {}
    )
)

print("\nText:")
print(
    enriched_results[0].get(
        "text",
        ""
    )[:1000]
)

ENRICHED RESULT STRUCTURE

Keys:
dict_keys(['chunk_id', 'rrf_score', 'dense_rank', 'bm25_rank', 'dense_score', 'bm25_score', 'hybrid_rank', 'text', 'metadata'])

Metadata:
{'experiment_id': 'SUMMARY-OXYGEN', 'doc_type': 'summary', 'source_investigation': 'PSI-159 (CFI-G)', 'chunk_type': 'dataset_summary', 'source': 'SUMMARY-OXYGEN.md'}

Text:
# Effect of oxygen concentration — all tests

## Summary

- **Air-like oxygen (18-23%)** (64 tests, O2 18.0-23.4%): 41 ignited, 41 self-extinguished, 0 still burning when the fuel flow was cut; median burn time 8.0 s, longest 20.0 s; mean hazard index 10.9/100. Longest-burning tests: CFI-21106-G6, CFI-21106-G7, CFI-21162-C1.
- **Enriched oxygen (33-40%)** (115 tests, O2 33.2-40.0%): 91 ignited, 61 self-extinguished, 30 still burning when the fuel flow was cut; median burn time 50.0 s, longest 304.0 s; mean hazard index 34.8/100. Longest-burning tests: CFI-21162-A4, CFI-21153-C4, CFI-21153-A5.

At air-like oxygen every lit flame went out by itself;

In [4]:
# ============================================================
# TEST 30 QUESTIONS
# HYBRID + ENRICHMENT + CROSS-ENCODER
# ============================================================

def test_30_questions():

    TEST_QUERIES = [
        "How does oxygen concentration affect flame behavior in microgravity?",
        "What happens to burn time when oxygen concentration changes?",
        "Do flames self-extinguish at 21% oxygen?",
        "How does pressure affect flame behavior?",
        "Were low-pressure tests safer than high-pressure tests?",
        "How does fuel flow affect burn time?",
        "What happens to the flame when the fuel flow increases?",
        "How hot did the burner get during the tests?",
        "What is the relationship between peak burner temperature and burn time?",
        "Which conditions led to hot ignition?",
        "Why did some tests fail to ignite?",
        "What factors influence burn time?",
        "Which experiments had the longest burn times?",
        "Which tests were still burning when the fuel flow was cut?",
        "Which tests self-extinguished?",
        "Were cool flames observed in the experiments?",
        "Which tests recorded a cool flame?",
        "Which fuels were tested in the experiments?",
        "How does the fuel type affect flame behavior?",
        "Did butane flames burn longer than ethane flames?",
        "What is the effect of diluting the fuel with nitrogen?",
        "How do 30% butane flames behave?",
        "What conditions are associated with high fire risk?",
        "Which experiments had the highest hazard index?",
        "How is fire risk related to oxygen concentration?",
        "How do experimental conditions affect flame behavior?",
        "What conditions were used in the microgravity experiments?",
        "How do oxygen concentration and pressure affect burn time?",
        "How are oxygen concentration, burner temperature and burn time related?",
        "What experimental conditions lead to a flame that keeps burning?",
    ]

    print("=" * 90)
    print("30-QUESTION RETRIEVAL TEST")
    print("=" * 90)

    print(
        f"\nTotal questions: {len(TEST_QUERIES)}"
    )

    # ========================================================
    # COMPONENTS
    # ========================================================

    hybrid_retriever = HybridRetriever()

    reranker = CrossEncoderReranker()

    metadata_filter = MetadataFilter(
        chunks_file=str(CHUNKS_FILE)
    )

    # ========================================================
    # RESULTS
    # ========================================================

    all_results = []

    # ========================================================
    # RUN QUESTIONS
    # ========================================================

    for question_number, query in enumerate(
        TEST_QUERIES,
        start=1
    ):

        print("\n\n")

        print("=" * 90)

        print(
            f"QUESTION {question_number}/"
            f"{len(TEST_QUERIES)}"
        )

        print("=" * 90)

        print("\nQUERY:")

        print(query)

        # ====================================================
        # HYBRID
        # ====================================================

        hybrid_results = hybrid_retriever.search(
            query=query,
            dense_top_k=20,
            bm25_top_k=20,
            top_k=20
        )

        # ====================================================
        # ENRICH
        # ====================================================

        enriched_results = (
            metadata_filter.enrich_results(
                hybrid_results
            )
        )

        # ====================================================
        # HYBRID TOP 5
        # ====================================================

        print("\n" + "-" * 90)
        print("HYBRID TOP 5")
        print("-" * 90)

        for result in enriched_results[:5]:

            metadata = result.get(
                "metadata",
                {}
            )

            experiment_id = metadata.get(
                "experiment_id",
                "N/A"
            )

            material = metadata.get(
            "fuel_mix",
                "N/A"
            )

            print(
                f"Rank {result.get('hybrid_rank', 'N/A'):>2} | "
                f"Chunk {result.get('chunk_id', 'N/A'):>4} | "
                f"Experiment {experiment_id:<10} | "
                f"Fuel: {material}"
            )

            print(
                f"    RRF: "
                f"{result.get('rrf_score', 0):.6f}"
            )

        # ====================================================
        # CROSS-ENCODER
        # ====================================================

        reranked_results = reranker.rerank(
            query=query,
            candidates=enriched_results,
            top_k=5
        )

        # ====================================================
        # RERANKER RESULTS
        # ====================================================

        print("\n" + "-" * 90)
        print("CROSS-ENCODER TOP 5")
        print("-" * 90)

        for result in reranked_results:

            metadata = result.get(
                "metadata",
                {}
            )

            experiment_id = metadata.get(
                "experiment_id",
                "N/A"
            )

            material = metadata.get(
            "fuel_mix",
                "N/A"
            )

            print(
                f"Rank {result.get('reranker_rank', 'N/A'):>2} | "
                f"Chunk {result.get('chunk_id', 'N/A'):>4} | "
                f"Experiment {experiment_id:<10} | "
                f"Fuel: {material}"
            )

            print(
                f"    Score: "
                f"{result.get('reranker_score', 0):.4f} | "
                f"Hybrid Rank: "
                f"{result.get('hybrid_rank', 'N/A')}"
            )

        # ====================================================
        # TOP EVIDENCE
        # ====================================================

        print("\n" + "-" * 90)
        print("TOP RERANKED EVIDENCE")
        print("-" * 90)

        for rank, result in enumerate(
            reranked_results[:3],
            start=1
        ):

            metadata = result.get(
                "metadata",
                {}
            )

            experiment_id = metadata.get(
                "experiment_id",
                "N/A"
            )

            text = result.get(
                "text",
                ""
            )

            print(
                f"\n[{rank}] "
                f"Experiment {experiment_id} "
                f"(Chunk {result.get('chunk_id')})"
            )

            print(
                text[:700]
                .replace("\n", " ")
            )

            if len(text) > 700:
                print("...")

        # ====================================================
        # SAVE RESULTS
        # ====================================================

        all_results.append({
            "question_number": question_number,
            "query": query,
            "hybrid_results": enriched_results,
            "reranked_results": reranked_results
        })

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print("\n\n")

    print("=" * 90)
    print("30-QUESTION TEST COMPLETE")
    print("=" * 90)

    print(
        f"\nSuccessfully tested "
        f"{len(all_results)}/"
        f"{len(TEST_QUERIES)} questions."
    )

    return all_results

In [5]:
results_30 = test_30_questions()

30-QUESTION RETRIEVAL TEST

Total questions: 30


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  26%|██▌       | 51/199 [00:00<00:00, 500.96it/s]

Loading weights:  51%|█████▏    | 102/199 [00:00<00:00, 255.21it/s]

Loading weights:  70%|███████   | 140/199 [00:00<00:00, 291.87it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 367.83it/s]

... (32 outputs left out)


RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 717
Reranker Score: -1.0177
Hybrid Rank: 6
Dense Rank: None
BM25 Rank: 3

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 44
Reranker Score: -4.6919
Hybrid Rank: 7
Dense Rank: 4
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 192
Reranker Score: -5.2291
Hybrid Rank: 1
Dense Rank: 1
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 112
Reranker Score: -5.2296
Hybrid Rank: 11
Dense Rank: 6
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 196
Reranker Score: -5.2562
Hybrid Rank: 9
Dense Rank: 5
BM25 Rank: None

------------------------------------------------------------------------------------------
CROSS-ENCODER TOP 5
------------------------------------------------------------------------------------------
Rank  1 


RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 720
Reranker Score: 2.3832
Hybrid Rank: 1
Dense Rank: 1
BM25 Rank: 1

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 721
Reranker Score: -3.0771
Hybrid Rank: 2
Dense Rank: 2
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 41
Reranker Score: -6.4339
Hybrid Rank: 3
Dense Rank: None
BM25 Rank: 2

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 717
Reranker Score: -6.4573
Hybrid Rank: 4
Dense Rank: 3
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 45
Reranker Score: -6.4616
Hybrid Rank: 5
Dense Rank: None
BM25 Rank: 3

------------------------------------------------------------------------------------------
CROSS-ENCODER TOP 5
------------------------------------------------------------------------------------------
Rank  1 | Chun


RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 720
Reranker Score: 0.3882
Hybrid Rank: 2
Dense Rank: None
BM25 Rank: 1

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 26
Reranker Score: -1.3205
Hybrid Rank: 19
Dense Rank: 10
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 38
Reranker Score: -1.3893
Hybrid Rank: 9
Dense Rank: 5
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 30
Reranker Score: -1.4403
Hybrid Rank: 7
Dense Rank: 4
BM25 Rank: None

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 706
Reranker Score: -1.4935
Hybrid Rank: 15
Dense Rank: 8
BM25 Rank: None

------------------------------------------------------------------------------------------
CROSS-ENCODER TOP 5
------------------------------------------------------------------------------------------
Rank  1 |


RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 663
Reranker Score: 0.7438
Hybrid Rank: 13
Dense Rank: None
BM25 Rank: 7

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 659
Reranker Score: 0.6954
Hybrid Rank: 11
Dense Rank: None
BM25 Rank: 6

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 667
Reranker Score: 0.6610
Hybrid Rank: 15
Dense Rank: None
BM25 Rank: 8

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 671
Reranker Score: 0.5277
Hybrid Rank: 17
Dense Rank: None
BM25 Rank: 9

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 679
Reranker Score: 0.5167
Hybrid Rank: 19
Dense Rank: None
BM25 Rank: 10

------------------------------------------------------------------------------------------
CROSS-ENCODER TOP 5
------------------------------------------------------------------------------------------
Rank  1